# SupplyForge AI 2.0: KAN-35 - Hierarchical Aggregation Features



Tài liệu này thực hiện việc tính toán các đặc trưng chuỗi thời gian phân cấp (Hierarchical Time-Series Features) ở cấp độ Nhóm sản phẩm (Brand và Category) để hỗ trợ dự báo nhu cầu SKU có dữ liệu thưa thớt hoặc SKU mới (Cold-Start).



Các đặc trưng bao gồm lags (1, 7 ngày), rolling mean (7, 30 ngày), và doanh thu lũy kế (cumulative sales) ở cả cấp Brand và Category. Các đặc trưng này được bảo vệ chống rò rỉ dữ liệu (Target Leakage) bằng cách lùi ngày `shift(1)` trước mọi phép tính rolling và cumulative.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Bảo toàn số lượng dòng**: Kích thước lưới chuỗi thời gian không thay đổi (giữ nguyên 323,250 dòng).

2. **Chống rò rỉ dữ liệu**: Chỉ sử dụng lịch sử lùi ngày (`shift(1)`) của nhóm làm đầu vào cho rolling và cumulative.

3. **Tách biệt phân cấp**: Các phép tính đặc trưng được thực hiện độc lập cho từng nhóm phân cấp trước khi left join vào lưới.


In [1]:
import pandas as pd

import numpy as np

import os

import sys



# Liên kết thư mục scripts để import module

sys.path.append('/home/nguyen-dai-tung/supplyforge-ai/scripts')

from hierarchical_features import generate_hierarchical_features



DATA_DIR = '/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH'

grid_path = os.path.join(DATA_DIR, 'calendar_daily_sales_metadata.csv')



print("Đang tải lưới dữ liệu Daily Calendar-SKU...")

df_grid = pd.read_csv(grid_path)

print(f"Kích thước lưới ban đầu: {df_grid.shape}")


Đang tải lưới dữ liệu Daily Calendar-SKU...


Kích thước lưới ban đầu: (323250, 33)


In [2]:
print("Đang chạy quy trình tính toán đặc trưng phân cấp...")

df_featured = generate_hierarchical_features(df_grid)

print(f"Kích thước lưới sau khi tính toán: {df_featured.shape}")

print("Danh sách đặc trưng phân cấp mới thêm vào:")

print([col for col in df_featured.columns if 'brand_' in col or 'category_' in col])


Đang chạy quy trình tính toán đặc trưng phân cấp...


Tổng hợp và tính toán đặc trưng cấp Brand...
Tổng hợp và tính toán đặc trưng cấp Category...
Đang ghép nối các đặc trưng phân cấp trở lại lưới Daily SKU Grid...
Kích thước lưới sau khi tính toán: (323250, 43)
Danh sách đặc trưng phân cấp mới thêm vào:
['brand_encoded', 'brand_lag_1', 'brand_lag_7', 'brand_roll_mean_7', 'brand_roll_mean_30', 'brand_cumulative_sales', 'category_lag_1', 'category_lag_7', 'category_roll_mean_7', 'category_roll_mean_30', 'category_cumulative_sales']


In [3]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")



# 1. Xác thực số lượng dòng được bảo toàn

print(f"Số dòng trước: {len(df_grid)} | Số dòng sau: {len(df_featured)}")

assert len(df_grid) == len(df_featured), "Lỗi: Số lượng dòng lưới bị thay đổi!"

print("1. Đảm bảo bảo toàn số dòng lưới: Đạt")



# 2. Kiểm chứng thủ công toán học trên một Brand và một Category mẫu

sample_brand = df_featured['brand'].dropna().unique()[0]

sample_cat = df_featured['categoryId'].dropna().unique()[0]



print(f"Kiểm chứng thủ công Brand: {sample_brand}")

# Tính thực tế từ dữ liệu đầu vào của Brand

df_brand_actual = df_grid[df_grid['brand'] == sample_brand].groupby('date')['quantity'].sum().reset_index()

df_brand_actual = df_brand_actual.sort_values('date').reset_index(drop=True)



# Lấy kết quả đặc trưng từ dataframe sau tính toán (1 dòng đại diện mỗi ngày)

df_brand_featured = df_featured[df_featured['brand'] == sample_brand].drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)



test_idx = 42

qty_history = df_brand_actual['quantity'].iloc[:test_idx].values



expected_lag1 = qty_history[-1]

expected_sum7 = sum(qty_history[-7:])

expected_mean7 = expected_sum7 / 7.0

expected_cumulative = sum(qty_history)



actual_lag1 = df_brand_featured['brand_lag_1'].iloc[test_idx]

actual_mean7 = df_brand_featured['brand_roll_mean_7'].iloc[test_idx]

actual_cumulative = df_brand_featured['brand_cumulative_sales'].iloc[test_idx]



print(f" - lag_1: Thực tế={actual_lag1}, Kỳ vọng={expected_lag1}")

print(f" - roll_mean_7: Thực tế={actual_mean7}, Kỳ vọng={expected_mean7}")

print(f" - cumulative_sales: Thực tế={actual_cumulative}, Kỳ vọng={expected_cumulative}")



assert abs(actual_lag1 - expected_lag1) < 1e-5, "Lỗi đối chiếu brand_lag_1!"

assert abs(actual_mean7 - expected_mean7) < 1e-5, "Lỗi đối chiếu brand_roll_mean_7!"

assert abs(actual_cumulative - expected_cumulative) < 1e-5, "Lỗi đối chiếu brand_cumulative_sales!"

print("2. Đối chiếu kiểm chứng mẫu Brand hoàn toàn chính xác: Đạt\n")



print(f"Kiểm chứng thủ công Category: {sample_cat}")

df_cat_actual = df_grid[df_grid['categoryId'] == sample_cat].groupby('date')['quantity'].sum().reset_index()

df_cat_actual = df_cat_actual.sort_values('date').reset_index(drop=True)



df_cat_featured = df_featured[df_featured['categoryId'] == sample_cat].drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)



qty_cat_history = df_cat_actual['quantity'].iloc[:test_idx].values



expected_cat_lag1 = qty_cat_history[-1]

expected_cat_sum7 = sum(qty_cat_history[-7:])

expected_cat_mean7 = expected_cat_sum7 / 7.0

expected_cat_cumulative = sum(qty_cat_history)



actual_cat_lag1 = df_cat_featured['category_lag_1'].iloc[test_idx]

actual_cat_mean7 = df_cat_featured['category_roll_mean_7'].iloc[test_idx]

actual_cat_cumulative = df_cat_featured['category_cumulative_sales'].iloc[test_idx]



print(f" - lag_1: Thực tế={actual_cat_lag1}, Kỳ vọng={expected_cat_lag1}")

print(f" - roll_mean_7: Thực tế={actual_cat_mean7}, Kỳ vọng={expected_cat_mean7}")

print(f" - cumulative_sales: Thực tế={actual_cat_cumulative}, Kỳ vọng={expected_cat_cumulative}")



assert abs(actual_cat_lag1 - expected_cat_lag1) < 1e-5, "Lỗi đối chiếu category_lag_1!"

assert abs(actual_cat_mean7 - expected_cat_mean7) < 1e-5, "Lỗi đối chiếu category_roll_mean_7!"

assert abs(actual_cat_cumulative - expected_cat_cumulative) < 1e-5, "Lỗi đối chiếu category_cumulative_sales!"

print("3. Đối chiếu kiểm chứng mẫu Category hoàn toàn chính xác: Đạt\n")


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

Số dòng trước: 323250 | Số dòng sau: 323250
1. Đảm bảo bảo toàn số dòng lưới: Đạt
Kiểm chứng thủ công Brand: Bear


 - lag_1: Thực tế=1.0, Kỳ vọng=1
 - roll_mean_7: Thực tế=2.4285714285714284, Kỳ vọng=2.4285714285714284
 - cumulative_sales: Thực tế=86.0, Kỳ vọng=86
2. Đối chiếu kiểm chứng mẫu Brand hoàn toàn chính xác: Đạt

Kiểm chứng thủ công Category: 669f2376ef600c835295346a
 - lag_1: Thực tế=0.0, Kỳ vọng=0
 - roll_mean_7: Thực tế=1.5714285714285714, Kỳ vọng=1.5714285714285714
 - cumulative_sales: Thực tế=47.0, Kỳ vọng=47
3. Đối chiếu kiểm chứng mẫu Category hoàn toàn chính xác: Đạt



In [4]:
print("=== LƯU ĐÈ LƯỚI THỜI GIAN ĐÃ ĐƯỢC LÀM GIÀU ĐẶC TRƯNG PHÂN CẤP ===")



# Bảo toàn các cột category để tránh mất định dạng phân loại của LightGBM

for col in ['brand_encoded', 'categoryId_encoded', 'warehouse_encoded']:

    df_featured[col] = df_featured[col].astype('category')



df_featured.to_csv(grid_path, index=False)

print(f"Đã lưu thành công lưới dữ liệu đặc trưng phân cấp tại: {grid_path}")


=== LƯU ĐÈ LƯỚI THỜI GIAN ĐÃ ĐƯỢC LÀM GIÀU ĐẶC TRƯNG PHÂN CẤP ===


Đã lưu thành công lưới dữ liệu đặc trưng phân cấp tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/calendar_daily_sales_metadata.csv
